In [0]:
import urllib.request
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
destination = "/Volumes/nyc_taxi/bronze/landing/yellow_tripdata_2024-01.parquet"
urllib.request.urlretrieve(url, destination)
print ("File downloaded")

In [0]:
%fs ls /Volumes/nyc_taxi/bronze/landing/

In [0]:
display(dbutils.fs.ls("/Volumes/nyc_taxi/bronze/landing/"))

In [0]:
from pyspark.sql import functions as F

#Paths
landing_path = "/Volumes/nyc_taxi/bronze/landing/"
checkpoint_path = "/Volumes/nyc_taxi/bronze/checkpoints/yellow_trips/"
schema_locations = "/Volumes/nyc_taxi/bronze/schemas/yellow_trips/"

In [0]:

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_locations)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

(
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("nyc_taxi.bronze.yellow_trips")
)
